# Multimodal Meme Understanding: Image, Text, or Both?



## 1. Environment Setup & Google Drive Mount
Mount Google Drive, configure data and output paths, and install the project dependencies.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os

BRANCH = os.environ.get('BRANCH', 'main')
REPO_URL = 'https://github.com/arcrek/dl2026.git'
ROOT = '/content/drive/MyDrive/dl2026_memotion'
os.environ['DATA_DIR'] = '/content/data/memotion'
os.environ['FEATURE_DIR'] = f'{ROOT}/features'
os.environ['RESULTS_DIR'] = f'{ROOT}/results'

for d in ('features', 'results'):
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

if os.path.exists('/content/repo'):
    !git -C /content/repo fetch && git -C /content/repo checkout $BRANCH && git -C /content/repo pull
else:
    !git clone -b $BRANCH $REPO_URL /content/repo
%cd /content/repo

!pip install -q transformers scikit-learn pyarrow huggingface_hub pytest pandas torchvision


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already on 'main'
Your branch is up to date with 'origin/main'.
Already up to date.
/content/repo


## 2. Dataset Download & Verification
Restore or download Memotion 7k, verify the image files, and cache the dataset on Google Drive.
For text metadata only, pass `--jsonl-only` to `scripts/download_data.sh` to skip downloading images.


In [3]:
TAR = f'{ROOT}/memotion.tar'
if os.path.exists(TAR) and not os.path.exists('/content/data/memotion/train.jsonl'):
    !mkdir -p /content/data && tar -xf "$TAR" -C /content/data

!bash scripts/download_data.sh
!python src/data.py --images

if not os.path.exists(TAR):
    !tar -cf "$TAR" -C /content/data memotion


train.jsonl already exists, skipping download...
validation.jsonl already exists, skipping download...
test.jsonl already exists, skipping download...
Dataset ready in /content/data/memotion
train       n= 5593 labels={0: 518, 1: 1762, 2: 3313} OK
  All 5593 images verified for train.
validation  n=  699 labels={0: 59, 1: 221, 2: 419} OK
  All 699 images verified for validation.
test        n=  700 labels={0: 54, 1: 218, 2: 428} OK
  All 700 images verified for test.
Holdout: fit=5034 holdout=559 dist={0: 54, 1: 165, 2: 340}


## 3. Data Preparation & Preprocessing

### Label Normalization
Raw sentiment labels in Memotion 7k follow a 5-point scale:
`very_positive`, `positive`, `neutral`, `negative`, `very_negative`

For standard 3-class sentiment classification, they are consolidated into:
- `positive` (from very positive, positive, pos)
- `neutral` (from neutral, neu)
- `negative` (from very negative, negative, neg)


In [4]:
import os, sys
sys.path.insert(0, "src")
import pandas as pd
from PIL import Image
from preprocessing import (
    normalize_sentiment,
    is_valid_image,
    get_image_path,
    load_raw_memotion_df,
    LABEL2ID,
    ID2LABEL,
)

def normalize_sentiment(label):
    if pd.isna(label):
        return None

    label = str(label).strip().lower()
    label = label.replace("_", " ")

    if label in ["very positive", "positive", "pos"]:
        return "positive"
    if label in ["neutral", "neu"]:
        return "neutral"
    if label in ["very negative", "negative", "neg"]:
        return "negative"

    return None


### 3.1. Create Clean DataFrame
After this step, the dataset contains only 3 primary columns:
`text`, `image_path`, `label_name`

Samples missing valid image files or sentiment labels are dropped.


In [5]:
# Load DataFrame (supports Kaggle df, Colab, or local jsonl)
if "df" not in locals():
    df = load_raw_memotion_df(os.environ.get("DATA_DIR", "data/memotion"))

data = pd.DataFrame()
text_col = "text_ocr" if "text_ocr" in df.columns else ("text_corrected" if "text_corrected" in df.columns else "text")
image_col = "image_name" if "image_name" in df.columns else ("img" if "img" in df.columns else "image")
sentiment_col = "overall_sentiment" if "overall_sentiment" in df.columns else ("raw_sentiment" if "raw_sentiment" in df.columns else "label")

data["text"] = df[text_col].fillna("").astype(str)
data["image_path"] = df[image_col].apply(lambda x: get_image_path(x, base_dir=os.environ.get("DATA_DIR", "data/memotion")))
data["label_name"] = df[sentiment_col].apply(normalize_sentiment)

print("Before filtering invalid samples:", len(data))
print("Samples with found images:", data["image_path"].notna().sum())
print("Samples with valid labels:", data["label_name"].notna().sum())

# Drop samples missing image or label
data = data.dropna(subset=["image_path", "label_name"]).reset_index(drop=True)

print("After dropping missing image/label:", len(data))
print(data["label_name"].value_counts())
data.head()


Before filtering invalid samples: 6992
Samples with found images: 6992
Samples with valid labels: 6992
After dropping missing image/label: 6992
label_name
positive    4160
neutral     2201
negative     631
Name: count, dtype: int64


,text,image_path,label_name
0,LOOK THERE MY FRIEND LIGHTYEAR NOW ALL SOHALIK...,/content/data/memotion/img/train_00000.png,positive
1,FACEBOOK '10 YEAR CHALLENGE': A PLOY OR A SIMP...,/content/data/memotion/img/train_00001.png,positive
2,Drink water you may not meme-generator.com,/content/data/memotion/img/train_00002.png,neutral
3,RT @BehindScenesPic: Martin Scorsese and Leona...,/content/data/memotion/img/train_00003.png,neutral
4,Russian Leonardo DiCaprio omg,/content/data/memotion/img/train_00004.png,positive


### 3.2. Filter Corrupted Images Before Training
Some images in Memotion 7k may be corrupted or truncated. If not filtered beforehand, the DataLoader may crash during training with:
```
OSError: image file is truncated
```
Therefore, each image file is verified once before training.


In [6]:
def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()

        with Image.open(path) as img:
            img.convert("RGB")

        return True
    except Exception:
        return False

before_filter = len(data)
data = data[data["image_path"].apply(is_valid_image)].reset_index(drop=True)
after_filter = len(data)

print("Corrupted images dropped:", before_filter - after_filter)
print("Final sample count for training:", after_filter)
print(data["label_name"].value_counts())
data.head()


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Corrupted images dropped: 1
Final sample count for training: 6991
label_name
positive    4159
neutral     2201
negative     631
Name: count, dtype: int64


,text,image_path,label_name
0,LOOK THERE MY FRIEND LIGHTYEAR NOW ALL SOHALIK...,/content/data/memotion/img/train_00000.png,positive
1,FACEBOOK '10 YEAR CHALLENGE': A PLOY OR A SIMP...,/content/data/memotion/img/train_00001.png,positive
2,Drink water you may not meme-generator.com,/content/data/memotion/img/train_00002.png,neutral
3,RT @BehindScenesPic: Martin Scorsese and Leona...,/content/data/memotion/img/train_00003.png,neutral
4,Russian Leonardo DiCaprio omg,/content/data/memotion/img/train_00004.png,positive


### 3.3. Encode Labels to Numeric IDs
PyTorch models require numeric target labels rather than strings:
- `negative`: 0
- `neutral`: 1
- `positive`: 2


In [7]:
label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

data["label"] = data["label_name"].map(label2id)
data.head()


,text,image_path,label_name,label
0,LOOK THERE MY FRIEND LIGHTYEAR NOW ALL SOHALIK...,/content/data/memotion/img/train_00000.png,positive,2
1,FACEBOOK '10 YEAR CHALLENGE': A PLOY OR A SIMP...,/content/data/memotion/img/train_00001.png,positive,2
2,Drink water you may not meme-generator.com,/content/data/memotion/img/train_00002.png,neutral,1
3,RT @BehindScenesPic: Martin Scorsese and Leona...,/content/data/memotion/img/train_00003.png,neutral,1
4,Russian Leonardo DiCaprio omg,/content/data/memotion/img/train_00004.png,positive,2


## 4. Text-only Baseline (BERT)
Fine-tune `bert-base-uncased` for 3-class sentiment across seeds 0, 1, and 2.
Load the training helpers below, then train and summarize the results.


### 4.1 Loss Function & Optimizer Setup
Load the BERT classifier and the helper for configuring loss and differential learning rates.


In [8]:
import sys
sys.path.insert(0, 'src')
import torch
import torch.nn as nn
from text import BertMemeClassifier, build_loss_and_optimizer

print('Part 1: Loss & Optimizer module loaded.')


Part 1: Loss & Optimizer module loaded.


### 4.2 Evaluation Function
Load `evaluate` to measure classification performance on a dataset split.


In [9]:
from text import evaluate

print('Part 2: Evaluation module loaded.')


Part 2: Evaluation module loaded.


### 4.3 Training Step (Single Epoch)
Load `train_epoch` for one pass of training batches and optimizer updates.


In [10]:
from text import train_epoch

print('Part 3: Single-epoch training step module loaded.')


Part 3: Single-epoch training step module loaded.


### 4.4 Training Loop
Load `train_loop` to coordinate training over multiple epochs.


In [11]:
from text import train_loop

print('Part 4: Multi-epoch training loop module loaded.')


Part 4: Multi-epoch training loop module loaded.


### 4.5 Training Execution
Train BERT for 5 epochs with batch size 32 across seeds 0, 1, and 2.


In [12]:
!python src/text.py --seeds 0 1 2 --epochs 5 --batch 32


Device: CUDA
Config: {'epochs': 5, 'batch': 32, 'lr_backbone': 1.5e-05, 'lr_head': 0.0005, 'max_length': 128, 'workers': 2, 'dropout': 0.2, 'weight_decay': 0.01}
Seeds: [0, 1, 2]
config.json: 100% 570/570 [00:00<00:00, 5.59MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 621kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 3.39MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 5.90MB/s]

model.safetensors: downloading bytes:  67% 293M/440M [00:00<00:00, 496MB/s, 21.3MB/s  ]
model.safetensors: downloading bytes:  82% 362M/440M [00:01<00:00, 549MB/s, 28.3MB/s  ]
model.safetensors: downloading bytes: 100% 415M/415M [00:01<00:00, 319MB/s, 39.2MB/s  ]
model.safetensors: reconstructing file: 100% 440M/440M [00:01<00:00, 338MB/s, 42.6MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 14239.81it/s]
[Seed 0 | Ep 01/05] Batch 030/158 ( 19.0%) | Loss: 1.1275
[Seed 0 | Ep 01/05] Batch 060/158 ( 38.0%) | Loss: 1.1144
[Seed 0 | Ep 01/05] Batch 090/158 ( 57.0%) | Loss: 1.1126
[Seed 0 | Ep 01/05

### 4.6 Verification & Contract Tests
Check the saved result schema with `tests/test_results.py`.


In [13]:
!pytest tests/test_results.py -v


============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/repo
plugins: langsmith-0.14.2, anyio-4.15.1, platformdirs-4.12.2, typeguard-4.6.0
collected 3 items                                                              

tests/test_results.py::test_roundtrip PASSED                             [ 33%]
tests/test_results.py::test_rejects_bad_probs_and_lengths PASSED         [ 66%]
tests/test_results.py::test_config PASSED                                [100%]

============================== 3 passed in 0.05s ===============================


### 4.7 Results Summary
Display holdout Macro-F1, test Accuracy, and test Macro-F1 for each seed, with mean and standard deviation.


In [14]:
import os, sys
sys.path.insert(0, 'src')
import numpy as np
import pandas as pd
from results import load_results

records = []
for s in [0, 1, 2]:
    try:
        res = load_results('text', s)
    except FileNotFoundError:
        continue
    records.append({
        'Seed': s,
        'Best Epoch (Holdout)': res['extra'].get('best_epoch', 'N/A'),
        'Holdout Macro-F1': round(res['extra'].get('holdout_macro_f1', 0.0), 4),
        'Test Accuracy': round(res['accuracy'], 4),
        'Test Macro-F1': round(res['macro_f1'], 4)
    })

if records:
    df = pd.DataFrame(records)
    if len(records) > 1:
        mean_row = pd.DataFrame([{
            'Seed': 'Mean +/- Std',
            'Best Epoch (Holdout)': '-',
            'Holdout Macro-F1': f"{df['Holdout Macro-F1'].mean():.4f} +/- {df['Holdout Macro-F1'].std():.4f}",
            'Test Accuracy': f"{df['Test Accuracy'].mean():.4f} +/- {df['Test Accuracy'].std():.4f}",
            'Test Macro-F1': f"{df['Test Macro-F1'].mean():.4f} +/- {df['Test Macro-F1'].std():.4f}"
        }])
        summary_df = pd.concat([df, mean_row], ignore_index=True)
    else:
        summary_df = df
    display(summary_df)
else:
    print('No results found. Run training cell above first.')


,Seed,Best Epoch (Holdout),Holdout Macro-F1,Test Accuracy,Test Macro-F1
0,0,1,0.3155,0.5629,0.2975
1,1,5,0.319,0.3957,0.3305
2,2,5,0.3295,0.4129,0.3293
3,Mean +/- Std,-,0.3213 +/- 0.0073,0.4572 +/- 0.0920,0.3191 +/- 0.0187


## 5. Image-only Baseline (ResNet50)
Load the pretrained ResNet50 classifier and image training helpers.
Training first fits the classification head, then fine-tunes the backbone across seeds 0, 1, and 2.


In [15]:
import os, sys
sys.path.insert(0, 'src')
from pathlib import Path
import json
import numpy as np
import torch
import torch.nn as nn
from models.image import (
    ImageArgs,
    ImageModel,
    MemeDataset,
    ResizeAndPad,
    evaluate,
    run_training,
    train_epoch,
    train_seed,
    valid_image,
)

print("Image Baseline (ResNet50) modules loaded successfully.")


Image Baseline (ResNet50) modules loaded successfully.


### 5.1 Training Execution
Set the dataset paths and training parameters, then run 3 head-only epochs and 5 fine-tuning epochs per seed.


In [16]:
from types import SimpleNamespace

# Hardcoded arguments instead of main() function / argparse
args = SimpleNamespace(
    seeds=[0, 1, 2],
    batch_size=32,
    workers=2,
    head_epochs=3,
    finetune_epochs=5,
    data_dir=os.environ.get("DATA_DIR", "data/memotion"),
    holdout_file=os.path.join(os.environ.get("FEATURE_DIR", "features"), "train_holdout.json"),
    results_dir=os.environ.get("RESULTS_DIR", "results"),
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print("Training configuration (hardcoded arguments):")
for k, v in vars(args).items():
    print(f"  {k}: {v}")

# Execute 2-stage training across seeds
training_summary = run_training(args, device=device)


Device: cuda
Training configuration (hardcoded arguments):
  seeds: [0, 1, 2]
  batch_size: 32
  workers: 2
  head_epochs: 3
  finetune_epochs: 5
  data_dir: /content/data/memotion
  holdout_file: /content/drive/MyDrive/dl2026_memotion/features/train_holdout.json
  results_dir: /content/drive/MyDrive/dl2026_memotion/results
Notice: 1 corrupted/missing images identified in training rows.
Device: cuda
Image Baseline Configuration (hardcoded arguments):
  seeds: [0, 1, 2]
  batch_size: 32
  head_epochs: 3
  finetune_epochs: 5
  workers: 2
  data_dir: /content/data/memotion
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 239MB/s]


[Seed 0] head_only 1/3 | Loss: 1.1252 | Holdout Macro-F1: 0.3058
[Seed 0] head_only 2/3 | Loss: 1.0711 | Holdout Macro-F1: 0.2907
[Seed 0] head_only 3/3 | Loss: 1.0400 | Holdout Macro-F1: 0.3218
[Seed 0] fine_tune 1/5 | Loss: 1.1055 | Holdout Macro-F1: 0.2922
[Seed 0] fine_tune 2/5 | Loss: 1.0193 | Holdout Macro-F1: 0.3032
[Seed 0] fine_tune 3/5 | Loss: 0.9554 | Holdout Macro-F1: 0.2990
[Seed 0] fine_tune 4/5 | Loss: 0.8804 | Holdout Macro-F1: 0.3091
[Seed 0] fine_tune 5/5 | Loss: 0.7936 | Holdout Macro-F1: 0.3215


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[Seed 0] Saved result: /content/drive/MyDrive/dl2026_memotion/results/image_seed0.json | Test Macro-F1: 0.2819 | Test Acc: 0.3729
[Seed 1] head_only 1/3 | Loss: 1.1238 | Holdout Macro-F1: 0.3319
[Seed 1] head_only 2/3 | Loss: 1.0697 | Holdout Macro-F1: 0.3191
[Seed 1] head_only 3/3 | Loss: 1.0419 | Holdout Macro-F1: 0.2784
[Seed 1] fine_tune 1/5 | Loss: 1.1066 | Holdout Macro-F1: 0.2834
[Seed 1] fine_tune 2/5 | Loss: 1.0172 | Holdout Macro-F1: 0.3137
[Seed 1] fine_tune 3/5 | Loss: 0.9504 | Holdout Macro-F1: 0.2990
[Seed 1] fine_tune 4/5 | Loss: 0.8847 | Holdout Macro-F1: 0.3129
[Seed 1] fine_tune 5/5 | Loss: 0.7935 | Holdout Macro-F1: 0.3142


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[Seed 1] Saved result: /content/drive/MyDrive/dl2026_memotion/results/image_seed1.json | Test Macro-F1: 0.3171 | Test Acc: 0.4600
[Seed 2] head_only 1/3 | Loss: 1.1247 | Holdout Macro-F1: 0.2746
[Seed 2] head_only 2/3 | Loss: 1.0717 | Holdout Macro-F1: 0.3249
[Seed 2] head_only 3/3 | Loss: 1.0426 | Holdout Macro-F1: 0.2356
[Seed 2] fine_tune 1/5 | Loss: 1.1064 | Holdout Macro-F1: 0.3005
[Seed 2] fine_tune 2/5 | Loss: 1.0242 | Holdout Macro-F1: 0.3124
[Seed 2] fine_tune 3/5 | Loss: 0.9561 | Holdout Macro-F1: 0.3021
[Seed 2] fine_tune 4/5 | Loss: 0.8755 | Holdout Macro-F1: 0.3174
[Seed 2] fine_tune 5/5 | Loss: 0.7926 | Holdout Macro-F1: 0.3002


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[Seed 2] Saved result: /content/drive/MyDrive/dl2026_memotion/results/image_seed2.json | Test Macro-F1: 0.3292 | Test Acc: 0.4886

IMAGE BASELINE TRAINING COMPLETED
Macro-F1 per seed: [0.2819, 0.3171, 0.3292]
Accuracy per seed: [0.3729, 0.46, 0.4886]
Macro-F1 mean ± std: 0.3094 ± 0.0201
Accuracy mean ± std: 0.4405 ± 0.0492


### 5.2 Verification & Contract Tests
Check the image baseline and saved result schema with `tests/test_image.py` and `tests/test_results.py`.


In [17]:
!pytest tests/test_results.py tests/test_image.py -v


============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/repo
plugins: langsmith-0.14.2, anyio-4.15.1, platformdirs-4.12.2, typeguard-4.6.0
collected 7 items                                                              

tests/test_results.py::test_roundtrip PASSED                             [ 14%]
tests/test_results.py::test_rejects_bad_probs_and_lengths PASSED         [ 28%]
tests/test_results.py::test_config PASSED                                [ 42%]
tests/test_image.py::test_resize_and_pad PASSED                          [ 57%]
tests/test_image.py::test_valid_image PASSED                             [ 71%]
tests/test_image.py::test_image_args_defaults PASSED                     [ 85%]
tests/test_image.py::test_image_args_custom PASSED                       [100%]

============================== 7 passed in 3.11s ======================

### 5.3 Results Summary
Display holdout Macro-F1, test Accuracy, and test Macro-F1 for each seed, with mean and standard deviation.


In [18]:
import os, sys
sys.path.insert(0, 'src')
import pandas as pd
from results import load_results

records = []
for s in [0, 1, 2]:
    try:
        res = load_results('image', s)
    except FileNotFoundError:
        continue
    records.append({
        'Seed': s,
        'Best Epoch (Holdout)': res['extra'].get('best_epoch', 'N/A'),
        'Holdout Macro-F1': round(res['extra'].get('holdout_macro_f1', 0.0), 4),
        'Test Accuracy': round(res['accuracy'], 4),
        'Test Macro-F1': round(res['macro_f1'], 4)
    })

if records:
    df = pd.DataFrame(records)
    if len(records) > 1:
        mean_row = pd.DataFrame([{
            'Seed': 'Mean ± Std',
            'Best Epoch (Holdout)': '-',
            'Holdout Macro-F1': f"{df['Holdout Macro-F1'].mean():.4f} ± {df['Holdout Macro-F1'].std():.4f}",
            'Test Accuracy': f"{df['Test Accuracy'].mean():.4f} ± {df['Test Accuracy'].std():.4f}",
            'Test Macro-F1': f"{df['Test Macro-F1'].mean():.4f} ± {df['Test Macro-F1'].std():.4f}"
        }])
        summary_df = pd.concat([df, mean_row], ignore_index=True)
    else:
        summary_df = df
    display(summary_df)
else:
    print('No results found. Run training cell above first.')


,Seed,Best Epoch (Holdout),Holdout Macro-F1,Test Accuracy,Test Macro-F1
0,0,3,0.3218,0.3729,0.2819
1,1,1,0.3319,0.46,0.3171
2,2,2,0.3249,0.4886,0.3292
3,Mean ± Std,-,0.3262 ± 0.0052,0.4405 ± 0.0603,0.3094 ± 0.0246


## 6. Multimodal Fusion (Concat, Cross-Attention & Product)
Combine BERT text features and ResNet50 image features using three fusion methods:
- **Concat:** Join text and image embeddings before classification.
- **Cross-Attention:** Let text tokens attend to spatial image features.
- **Product:** Project both embeddings to a shared dimension and multiply them element-wise.

Train across seeds 0, 1, and 2, and compare the saved results.


### 6.1 Model Architectures & Training Helpers
Load the three fusion models, dataset utilities, and training configuration helpers.


In [19]:
import sys
sys.path.insert(0, 'src')
import torch
import torch.nn as nn
from fusion import (
    ConcatFusionModel,
    CrossAttentionFusionModel,
    ProductFusionModel,
    FusionArgs,
    MultimodalMemeDataset,
    build_fusion_model,
    make_fusion_collate_fn,
    evaluate_split,
    train_seed,
    run_training,
    FUSION_CONFIGS,
    CrossAttentionArgs,
)
try:
    from cross_attn import CrossAttentionArgs
except ImportError:
    pass

print(f'Part 1: All 3 Multimodal Fusion architectures loaded. Configs: {FUSION_CONFIGS}')


Part 1: All 3 Multimodal Fusion architectures loaded. Configs: ['both_concat', 'both_cross_attn', 'both_product']


### 6.2 Evaluation Function
Load `evaluate_split` to evaluate a multimodal model on a dataset split.


In [20]:
from fusion import evaluate_split

print('Part 2: Multimodal evaluation module loaded.')


Part 2: Multimodal evaluation module loaded.


### 6.3 Per-Seed Training
Load `train_seed` to train and evaluate a fusion model for one random seed.


In [21]:
from fusion import train_seed

print('Part 3: Multimodal training & optimization module loaded.')


Part 3: Multimodal training & optimization module loaded.


### 6.4 Training Pipeline
Load `run_training` and the available fusion configurations before setting the training parameters.


In [22]:
from fusion import FusionArgs, run_training, FUSION_CONFIGS

print(f'Part 4: Multimodal fusion training pipeline loaded. Available configs: {FUSION_CONFIGS}')


Part 4: Multimodal fusion training pipeline loaded. Available configs: ['both_concat', 'both_cross_attn', 'both_product']


#### Training Parameters & Execution
Use `config="all"` for all three fusion methods, or select one configuration.
Train for 5 epochs per seed with backbone LR `1.5e-5`, head LR `5e-4`, and the image backbone frozen.


In [23]:
from types import SimpleNamespace

# Hardcoded arguments instead of main() function / argparse
# Set config="all" to train all 3 fusion methods, or specify: "both_concat", "both_cross_attn", "both_product"
args = SimpleNamespace(
    config="all",  # Trains all 3 fusion architectures: Concat, Cross-Attention, Product
    seeds=[0, 1, 2],
    epochs=5,
    batch_size=16,
    eval_batch_size=32,
    lr_backbone=1.5e-5,
    lr_head=5e-4,
    max_length=128,
    workers=2,
    dropout=0.2,
    weight_decay=0.01,
    freeze_image=True,
    data_dir=os.environ.get("DATA_DIR", "data/memotion"),
    holdout_file=os.path.join(os.environ.get("FEATURE_DIR", "features"), "train_holdout.json"),
    results_dir=os.environ.get("RESULTS_DIR", "results"),
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print("Multimodal Fusion Training configuration (hardcoded arguments):")
for k, v in vars(args).items():
    print(f"  {k}: {v}")

# Execute training for all 3 fusion architectures across seeds with preprocessing integrity checks
training_summary = run_training(args, device=device)


Device: cuda
Multimodal Fusion Training configuration (hardcoded arguments):
  config: all
  seeds: [0, 1, 2]
  epochs: 5
  batch_size: 16
  eval_batch_size: 32
  lr_backbone: 1.5e-05
  lr_head: 0.0005
  max_length: 128
  workers: 2
  dropout: 0.2
  weight_decay: 0.01
  freeze_image: True
  data_dir: /content/data/memotion
  holdout_file: /content/drive/MyDrive/dl2026_memotion/features/train_holdout.json
  results_dir: /content/drive/MyDrive/dl2026_memotion/results
Device: CUDA
Configurations to run: ['both_concat', 'both_cross_attn', 'both_product']
Seeds: [0, 1, 2]
Hyperparameters: {'epochs': 5, 'batch': 16, 'eval_batch': 32, 'lr_backbone': 1.5e-05, 'lr_head': 0.0005, 'max_length': 128, 'workers': 2, 'dropout': 0.2, 'weight_decay': 0.01, 'freeze_image': True, 'data_dir': '/content/data/memotion', 'feature_dir': '/content/drive/MyDrive/dl2026_memotion/features', 'holdout_file': '/content/drive/MyDrive/dl2026_memotion/features/train_holdout.json', 'results_dir': '/content/drive/MyDrive

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


=== Training [both_concat] Seed 0 on CUDA ===
[both_concat | Seed 0 | Ep 01/05] Batch 030/315 (  9.5%) | Loss: 1.1014
[both_concat | Seed 0 | Ep 01/05] Batch 060/315 ( 19.0%) | Loss: 1.1017
[both_concat | Seed 0 | Ep 01/05] Batch 090/315 ( 28.6%) | Loss: 1.1092
[both_concat | Seed 0 | Ep 01/05] Batch 120/315 ( 38.1%) | Loss: 1.1127
[both_concat | Seed 0 | Ep 01/05] Batch 150/315 ( 47.6%) | Loss: 1.1160
[both_concat | Seed 0 | Ep 01/05] Batch 180/315 ( 57.1%) | Loss: 1.1197
[both_concat | Seed 0 | Ep 01/05] Batch 210/315 ( 66.7%) | Loss: 1.1176
[both_concat | Seed 0 | Ep 01/05] Batch 240/315 ( 76.2%) | Loss: 1.1165
[both_concat | Seed 0 | Ep 01/05] Batch 270/315 ( 85.7%) | Loss: 1.1145
[both_concat | Seed 0 | Ep 01/05] Batch 300/315 ( 95.2%) | Loss: 1.1142
[both_concat | Seed 0 | Ep 01/05] Batch 315/315 (100.0%) | Loss: 1.1147
[both_concat | Seed 0] Epoch 01/05 | Train Loss: 1.1147 | Holdout F1: 0.2927 | Holdout Acc: 0.3918
[both_concat | Seed 0 | Ep 02/05] Batch 030/315 (  9.5%) | Los

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[both_concat | Seed 0] Saved result: /content/drive/MyDrive/dl2026_memotion/results/both_concat_seed0.json | Test Macro-F1: 0.3289 | Test Acc: 0.3986
[both_concat | Seed 1] Preprocessing: filtered 1 corrupted/missing images from fit set.


...

[both_product | Seed 1] Saved result: /content/drive/MyDrive/dl2026_memotion/results/both_product_seed1.json | Test Macro-F1: 0.3206 | Test Acc: 0.4071
[both_product | Seed 2] Preprocessing: filtered 1 corrupted/missing images from fit set.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


=== Training [both_product] Seed 2 on CUDA ===
[both_product | Seed 2 | Ep 01/05] Batch 030/315 (  9.5%) | Loss: 1.1541
[both_product | Seed 2 | Ep 01/05] Batch 060/315 ( 19.0%) | Loss: 1.1401
[both_product | Seed 2 | Ep 01/05] Batch 090/315 ( 28.6%) | Loss: 1.1378
[both_product | Seed 2 | Ep 01/05] Batch 120/315 ( 38.1%) | Loss: 1.1334
[both_product | Seed 2 | Ep 01/05] Batch 150/315 ( 47.6%) | Loss: 1.1373
[both_product | Seed 2 | Ep 01/05] Batch 180/315 ( 57.1%) | Loss: 1.1340
[both_product | Seed 2 | Ep 01/05] Batch 210/315 ( 66.7%) | Loss: 1.1338
[both_product | Seed 2 | Ep 01/05] Batch 240/315 ( 76.2%) | Loss: 1.1304
[both_product | Seed 2 | Ep 01/05] Batch 270/315 ( 85.7%) | Loss: 1.1271
[both_product | Seed 2 | Ep 01/05] Batch 300/315 ( 95.2%) | Loss: 1.1257
[both_product | Seed 2 | Ep 01/05] Batch 315/315 (100.0%) | Loss: 1.1239
[both_product | Seed 2] Epoch 01/05 | Train Loss: 1.1239 | Holdout F1: 0.3040 | Holdout Acc: 0.5921
[both_product | Seed 2 | Ep 02/05] Batch 030/315 

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[both_product | Seed 2] Saved result: /content/drive/MyDrive/dl2026_memotion/results/both_product_seed2.json | Test Macro-F1: 0.3231 | Test Acc: 0.3929

FINAL MULTIMODAL FUSION SUMMARY
[both_concat]
  Macro-F1 per seed: [0.3289, 0.3495, 0.3322] => mean: 0.3369 ± 0.0090
  Accuracy per seed: [0.3986, 0.4429, 0.4386] => mean: 0.4267 ± 0.0199
[both_cross_attn]
  Macro-F1 per seed: [0.3443, 0.3302, 0.3061] => mean: 0.3269 ± 0.0158
  Accuracy per seed: [0.4771, 0.3971, 0.5286] => mean: 0.4676 ± 0.0541
[both_product]
  Macro-F1 per seed: [0.3193, 0.3206, 0.3231] => mean: 0.3210 ± 0.0016
  Accuracy per seed: [0.4257, 0.4071, 0.3929] => mean: 0.4086 ± 0.0135


### 6.5 Verification & Contract Tests
Check fusion models, cross-attention, and saved result schemas with the three test files below.


In [24]:
!pytest tests/test_fusion.py tests/test_cross_attention.py tests/test_results.py -v


============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/repo
plugins: langsmith-0.14.2, anyio-4.15.1, platformdirs-4.12.2, typeguard-4.6.0
collected 59 items                                                             

tests/test_fusion.py::test_factory_defaults_kwargs_and_shapes[False-both_concat-ConcatFusionModel] PASSED [  1%]
tests/test_fusion.py::test_factory_defaults_kwargs_and_shapes[False-both_cross_attn-CrossAttentionFusionModel] PASSED [  3%]
tests/test_fusion.py::test_factory_defaults_kwargs_and_shapes[False-both_product-ProductFusionModel] PASSED [  5%]
tests/test_fusion.py::test_factory_defaults_kwargs_and_shapes[True-both_concat-ConcatFusionModel] PASSED [  6%]
tests/test_fusion.py::test_factory_defaults_kwargs_and_shapes[True-both_cross_attn-CrossAttentionFusionModel] PASSED [  8%]
tests/test_fusion.py::test_factory_defaults

### 6.6 Results Summary & Architecture Comparison
Compare holdout Macro-F1, test Accuracy, and test Macro-F1 across fusion methods, with mean and standard deviation over seeds.


In [25]:
import os, sys
sys.path.insert(0, 'src')
import numpy as np
import pandas as pd
from results import load_results

configs = ['both_concat', 'both_cross_attn', 'both_product']
config_labels = {
    'both_concat': 'Concat Fusion (3A)',
    'both_cross_attn': 'Cross-Attention (3B)',
    'both_product': 'Product Fusion (3C - Ablation)',
}

records = []
for cfg in configs:
    f1s, accs, ho_f1s = [], [], []
    for s in [0, 1, 2]:
        try:
            res = load_results(cfg, s)
            f1s.append(res['macro_f1'])
            accs.append(res['accuracy'])
            ho_f1s.append(res['extra'].get('holdout_macro_f1', 0.0))
        except FileNotFoundError:
            continue
    if f1s:
        records.append({
            'Architecture': config_labels.get(cfg, cfg),
            'Config Key': cfg,
            'Seeds Completed': f"{len(f1s)}/3",
            'Holdout Macro-F1': f"{np.mean(ho_f1s):.4f} ± {np.std(ho_f1s):.4f}",
            'Test Accuracy': f"{np.mean(accs):.4f} ± {np.std(accs):.4f}",
            'Test Macro-F1': f"{np.mean(f1s):.4f} ± {np.std(f1s):.4f}",
        })

if records:
    summary_df = pd.DataFrame(records)
    display(summary_df)
else:
    print('No results found. Run Multimodal Fusion training cell above first.')


,Architecture,Config Key,Seeds Completed,Holdout Macro-F1,Test Accuracy,Test Macro-F1
0,Concat Fusion (3A),both_concat,3/3,0.3388 ± 0.0208,0.4267 ± 0.0199,0.3369 ± 0.0090
1,Cross-Attention (3B),both_cross_attn,3/3,0.3312 ± 0.0122,0.4676 ± 0.0541,0.3269 ± 0.0158
2,Product Fusion (3C - Ablation),both_product,3/3,0.3447 ± 0.0068,0.4086 ± 0.0135,0.3210 ± 0.0016


## 7. Interactive Inference & Demo
This section provides an interactive pipeline to predict sentiment on any meme image and caption. It demonstrates:
1. **Single-sample prediction**: Predicts sentiment class (`negative`, `neutral`, `positive`) and shows probability distribution across classes.
2. **Modality comparison**: Evaluates Text-only (BERT), Image-only (ResNet-50), and Multimodal Fusion (Concat) side-by-side on the same input meme to visualize how textual vs visual cues influence sentiment.

In [ ]:
import os
import sys
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from transformers import AutoTokenizer

sys.path.insert(0, 'src')
from models.fusion import build_fusion_model, safe_open_image
from models.text import BertMemeClassifier, MODEL_NAME as BERT_MODEL_NAME
from models.image import ImageModel

CLASS_NAMES = ['negative', 'neutral', 'positive']
IMG_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_meme(image_path: str, caption: str, model_type: str = 'both_concat', device: str = None):
    """Run inference for a meme image and caption using the selected architecture."""
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
    
    if model_type in ['both_concat', 'both_cross_attn', 'both_product']:
        model = build_fusion_model(model_type, num_classes=3, freeze_image=True)
    elif model_type == 'text':
        model = BertMemeClassifier(model_name=BERT_MODEL_NAME, num_classes=3)
    elif model_type == 'image':
        model = ImageModel()
    else:
        raise ValueError(f'Unknown model_type: {model_type}')
        
    model.to(device)
    model.eval()
    
    # Process text
    enc = tokenizer([caption], padding=True, truncation=True, max_length=128, return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)
    
    # Process image
    if image_path and os.path.exists(image_path):
        img_pil = safe_open_image(image_path)
    else:
        img_pil = Image.new('RGB', (224, 224), (255, 255, 255))
    pixel_values = IMG_TF(img_pil).unsqueeze(0).to(device)
    
    with torch.no_grad():
        if model_type in ['both_concat', 'both_cross_attn', 'both_product']:
            logits = model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask)
        elif model_type == 'text':
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
        elif model_type == 'image':
            logits = model(pixel_values)
            
    probs = F.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
    pred_idx = int(probs.argmax())
    pred_class = CLASS_NAMES[pred_idx]
    return pred_class, {CLASS_NAMES[i]: float(probs[i]) for i in range(3)}, img_pil


In [ ]:
# Interactive Demo Execution
sample_image = 'data/memotion/img/test_00001.png'
sample_caption = 'THEY SAID I FIGHT LIKE A GIRL I TOOK THAT AS A COMPLIMENT'

print('--- Running Multimodal Meme Demo (Concat Fusion) ---')
pred_label, confidences, img_pil = predict_meme(
    image_path=sample_image,
    caption=sample_caption,
    model_type='both_concat'
)

print(f'Meme Caption        : "{sample_caption}"')
print(f'Predicted Sentiment : {pred_label.upper()}')
for cls_name, prob in confidences.items():
    print(f'  {cls_name:8s}: {prob*100:5.1f}%')

# Visualize
fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(9, 3.5))
ax_img.imshow(img_pil)
ax_img.axis('off')
ax_img.set_title('Input Meme Image', fontsize=10)

colors = ['#e74c3c', '#f39c12', '#2ecc71']
bars = ax_bar.barh(CLASS_NAMES, [confidences[c] * 100 for c in CLASS_NAMES], color=colors)
ax_bar.set_xlim(0, 100)
ax_bar.set_xlabel('Probability (%)')
ax_bar.set_title(f'Prediction: {pred_label.upper()} ({confidences[pred_label]*100:.1f}%)', fontweight='bold')
for bar in bars:
    w = bar.get_width()
    ax_bar.text(w + 1, bar.get_y() + bar.get_height()/2, f'{w:.1f}%', va='center')
plt.tight_layout()
plt.show()
